# Using Inline Functions with Semantic Kernel

This notebook demonstrates how to define and use inline functions with the Microsoft Semantic Kernel in .NET. You'll learn to create, configure, and invoke inline semantic functions for flexible AI-powered workflows.

**Step 1:** Install NuGet packages

Install the required NuGet packages for Semantic Kernel and DotNetEnv to enable AI-powered workflows and environment variable management.

In [23]:
// Import Semantic Kernel
#r "nuget: Microsoft.SemanticKernel, 1.23.0"
#r "nuget: DotNetEnv, 3.1.0"

Installed Packages DotNetEnv, 3.1.0 Microsoft.SemanticKernel, 1.23.0

**Step 2:** Load environment variables

Load environment variables from a `.env` file if present. This helps manage secrets and configuration settings for your application.

In [24]:
using DotNetEnv;
using System.IO;

var envFilePath = Path.Combine(Environment.CurrentDirectory, "../..", ".env");
if (File.Exists(envFilePath))
{
    Env.Load(envFilePath);
    Console.WriteLine($"Loaded environment variables from {envFilePath}");
}
else
{
    Console.WriteLine($"No .env file found at {envFilePath}");
}

Loaded environment variables from d:\personal\aiagent-workshop\notebooks\semantic-kernel-basic\../..\.env


**Step 3:** Instantiate the Semantic Kernel

Create and configure a Kernel instance, which will be used to interact with AI models.

In [25]:
using System.ClientModel;
using OpenAI;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using System.Text;

OpenAIClient client = null;
if(Environment.GetEnvironmentVariable("USE_AZURE_OPENAI") == "true")
{
    // Configure Azure OpenAI client
    var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(azureEndpoint) });
}
else if(Environment.GetEnvironmentVariable("USE_OPENAI") == "true")
{
    // Configure OpenAI client
    var apiKey = Environment.GetEnvironmentVariable("OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey));
}
else if(Environment.GetEnvironmentVariable("USE_GITHUB") == "true")
{
    // Configure GitHub model client
    var uri = Environment.GetEnvironmentVariable("GITHUB_MODEL_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("GITHUB_TOKEN");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(uri) });
}

var modelId = Environment.GetEnvironmentVariable("MODEL");
// Create a chat completion service
var builder = Kernel.CreateBuilder();
builder.AddOpenAIChatCompletion(modelId, client);

// Get the chat completion service
Kernel kernel = builder.Build();
var chat = kernel.GetRequiredService<IChatCompletionService>(); 

**Step 4:** Define an inline semantic function

Create an inline semantic function using a prompt template. This function will summarize input content, with a configurable word count.

In [26]:
string skPrompt = """
{{$input}}

Summarize the content above with a maximum of {{$wordCount}} words.

""";

**Step 5:** Create the function and configure prompt settings

Set up the execution settings for the inline function, such as maximum tokens, temperature, and top-p sampling. Then, create the function from the prompt template.

In [27]:
var promptTemplateConfig = new PromptTemplateConfig(skPrompt);
var promptTemplateFactory = new KernelPromptTemplateFactory();
var promptTemplate = promptTemplateFactory.Create(promptTemplateConfig);
var summaryFunction = kernel.CreateFunctionFromPrompt(skPrompt, executionSettings);

**Step 6:** Prepare input content

Set up some content to summarize. In this example, the input describes AI agents and their capabilities.

In [28]:
var input = """
AI agents are autonomous software programs that use artificial intelligence to perform tasks, make decisions, and interact with their environment. They can process data, learn from experience, and adapt their behavior to achieve specific goals. AI agents are used in applications such as virtual assistants, robotics, automated trading, and customer service chatbots. Their ability to operate independently and improve over time makes them valuable tools in a wide range of industries.
""";

**Step 7:** Call the inline function

Send the input and word count to the inline function and receive a summary response from the AI model.

In [29]:
// Call the kernel to get a response 
var response = await kernel.InvokeAsync(summaryFunction, new() { ["input"] = input, ["wordCount"] = 10 });  
Console.WriteLine($"Response: {response}");

Response: AI agents autonomously perform tasks, learn, adapt, and achieve goals.
